# Results figures and tables of the CLOAK methods paper

Every results figure (`fig02` to `fig14`) and every results table of the paper
is produced here from **synthetic data only**. The synthetic inputs come from
`synthetic_data/generate_synthetic_data.ipynb`; the code behind each cell is in
`figures/figlib.py`. Figures are written as PDF into `figures/`; tables as
LaTeX rows into `figures/results/`. A figure whose inputs do not exist yet
(`fig10` before the calibration batch has run, `fig14` before the other-band
tables exist) is skipped, not replaced by a placeholder.

* Cheap experiments run inline (seconds to minutes).
* The MCMC fits run once as `python -m cloak.mcmc_fit` subprocesses (about
  20-30 minutes each; `python figures/run_fits.py` runs them from a shell) and
  are cached under `figures/cache/` in directories keyed by a digest of their
  inputs, so re-running the notebook reuses them until a table, a data file or
  a fit option changes.
* The simulation-based calibration batch is a separate script
  (`python figures/run_sbc.py`, 50 draws by default at about half an hour
  each); the notebook plots whatever `figures/results/sbc_ranks.csv` contains.

Run it in a kernel of the environment that has the requirements installed
(here `henv`). Every number produced is the paper's: there is no reduced mode.

In [ ]:
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

HERE = os.path.abspath(os.getcwd())
ROOT = HERE if os.path.isdir(os.path.join(HERE, "cloak")) else os.path.dirname(HERE)
sys.path.insert(0, os.path.join(ROOT, "figures")); sys.path.insert(0, ROOT)
import figlib as L
L.style()
print("repository:", ROOT)
print("flux tables:", {b: os.path.relpath(p, ROOT) for b, p in L.available_tables().items()})

## Tables 3 and 4: fiducial systems and priors

In [ ]:
L.fiducial_table(); L.priors_table()
print(open(os.path.join(L.RESULTS, "tab_fiducial.tex")).read())

## Figure 2: wind density profiles

In [ ]:
fig = L.fig_wind_profiles(R=2.5); L.save_fig(fig, "fig02_wind_profiles"); plt.show()

## Figure 3: transformed integrand and quadrature convergence

Panel (b) compares the plain $n$-point Gauss--Legendre rule with adaptive
quadrature; the stars mark what the kernel actually computes at $n=16$,
including the piecewise rule for $\beta$-law rays grazing the photosphere.

In [ ]:
fig, quad_summary = L.fig_quadrature(); L.save_fig(fig, "fig03_quadrature"); plt.show()
for k, v in quad_summary.items():
    print(f"{k:24s} plain GL16 {v['gl16_plain']:.1e}   kernel {v['kernel']:.1e}")

## Figure 4: convergence with the phase step and the sector size

The coarser curve is interpolated onto the finest grid the way the fitters
interpolate the model onto the data phases, and the discrepancy is normalized
to the out-of-eclipse flux. Its **mean** over phase falls linearly with the
step; its **maximum** does not, because a point-like emitter enters totality
within $0.006^\circ$, a discontinuity that no linear interpolation resolves
(the maximum sits at the contact points and equals the flux just before
totality). The radial cell count is not varied: with the default 10 radial
cells a point-like emitter changes by $<10^{-9}$ and an extended one
($r = 1$--$2\,R_\odot$) by $<0.3\,\%$ against 40 cells, in a dozen partial-eclipse
phases only.

In [ ]:
study = L.convergence_study(force=True)   # always re-timed
fig = L.fig_convergence(study); L.save_fig(fig, "fig04_convergence"); plt.show()
for key in ("dth", "d2h"):
    for name, rows in study[key].items():
        print(key, name, [(r["value"], f"{r['max_rel_change']:.1e}", f"{r['wall_ms']:.1f} ms") for r in rows])

## Figure 5: per-cell versus mean-column attenuation (extended emitter)

A $6\,R_\odot$ emitting disk on System B's geometry ($R = 18\,R_\odot$, sectors of
$3^\circ$): attenuating each cell and averaging exceeds attenuating the
area-averaged column by a factor of tens in the partial phases (Jensen's
inequality), and the mean-column curve is unstable there because the mean
column jumps as cells cross the limb while the opaque limit of the table
amplifies every jump. The ratio in panel (c) is shown where the mean-column
flux exceeds $10^{-3}$ of the out-of-eclipse flux. On the compact System A the
same comparison never exceeds 8 % even for $r = 2.4\,R_\odot$.

In [ ]:
fig, percell = L.fig_percell(name="B", r=6.0); L.save_fig(fig, "fig05_percell"); plt.show(); print(percell)

## Figure 6: energy dependence of the eclipse

In [ ]:
fig, energy = L.fig_energy_dependence(); L.save_fig(fig, "fig06_energy_dependence"); plt.show()
for k, v in energy.items():
    print(f"{k:10s} width {v['width']:.4f}  depth {v['depth']:.3f}")

## Figure 7 and Table 5: the two exact invariances

In [ ]:
inv = L.invariance_study(); L.invariance_table(inv)
fig = L.fig_invariance(inv); L.save_fig(fig, "fig07_invariance"); plt.show()
print(open(os.path.join(L.RESULTS, "tab_invariance.tex")).read())

## Figures 8 and 9, Table 6: injection--recovery on System A

The fit is `python -m cloak.mcmc_fit` on the System A broad-band synthetic
light curve (command in `figlib.fit_config`): 32 walkers, 5000 steps, 1000
burn-in, jitter likelihood, `--kepler-mtot` with wind shape, opacity and floor
free. It runs once (about 25 minutes) and is cached. Bars and residuals use the
jitter likelihood's effective variance at the MAP, since System A carries 10 %
intrinsic variability by construction.

In [ ]:
cfg = L.ensure_fit("A_fiducial")
fit = L.load_fit(cfg)
truths = L.truth_for_names(fit["names"], rows_per_bin=float(np.median(fit["binned"]["n_points"])))
pred = L.predictive_curves(fit, n_draws=200)
fig, inj = L.fig_injection(fit, pred); L.save_fig(fig, "fig08_injection_fit"); plt.show(); print(inj)

In [ ]:
fig = L.fig_corner(fit, truths); L.save_fig(fig, "fig09_corner"); plt.show()
path, records = L.recovery_table(fit, truths)
print(open(path).read())

## Figure 10: simulation-based calibration

Produced from `figures/results/sbc_ranks.csv`, written by
`python figures/run_sbc.py` (50 draws by default; resumable; per-draw seeds;
about half an hour per draw). Until that file exists this cell prints a note
and writes no figure. The batch fits exactly the paper's model (jitter likelihood,
wind shape, opacity and floor free, `--kepler-mtot`) with two stated changes a
calibration test needs: the floor has a fixed prior (`figlib.floor_prior()`, 3 %
of System A's out-of-eclipse flux with a 2 % width) from which the injected
floor is drawn, and `q_m` is frozen at its drawn value
because its posterior equals its prior by construction. The jitter parameter's
reference value is only approximate (per-row variability diluted over the rows
of a bin), so its panel is a diagnostic and does not enter the verdict. Only
draws the fitter judged converged are used.

In [ ]:
fig, sbc = L.fig_sbc(); L.save_fig(fig, "fig10_sbc"); plt.show(); print(sbc)

## Figure 11: structure of the scale degeneracy

In [ ]:
ridge_fits = {}
for name in ("ridge_broad", "ridge_tightR", "ridge_fopa_frozen"):
    ridge_fits[name] = L.load_fit(L.ensure_fit(name))
fig = L.fig_ridge(ridge_fits, truths); L.save_fig(fig, "fig11_ridge"); plt.show()

## Figure 12: the profiled phase shift

In [ ]:
fig, shift_summary = L.fig_shift_profile(fit); L.save_fig(fig, "fig12_shift_profile"); plt.show(); print(shift_summary)

## Figure 13: bin estimators for Poisson data

In [ ]:
bias = L.binning_bias_study(trials=3000)
fig = L.fig_binning_bias(bias); L.save_fig(fig, "fig13_binning"); plt.show()

## Figure 14: cross-band prediction from the broad-band posterior

In [ ]:
fig, cross = L.fig_crossband(fit, n_draws=100); L.save_fig(fig, "fig14_crossband"); plt.show(); print(cross)

## Table 7: performance

In [ ]:
path, perf = L.performance_table(fit)
print(open(path).read())

## Outputs

In [ ]:
for d in (L.FIG_DIR, L.RESULTS):
    for f in sorted(os.listdir(d)):
        if f.endswith((".pdf", ".tex", ".json")):
            print(os.path.relpath(os.path.join(d, f), ROOT))